In [15]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

## Notes
https://www.kaggle.com/competitions/spaceship-titanic/overview

## Load Data

In [16]:
# train data
train_data = pd.read_csv("data/train.csv")
train_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [17]:
# test data
test_data = pd.read_csv("data/test.csv")
test_data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


## Exploratory Data Analysis (EDA)

In [18]:
def data_eda(data: pd.DataFrame, set_name: str):
    print(f"##### {set_name} Data EDA #####")
    print("Columns:\n", data.columns, "\n")
    print("Info:")
    data.info()
    print("\n")
    print("Describe:\n", data.describe(), "\n")
    print("Unique:\n", data.nunique(), "\n")
    print("Nulls:\n", data.isnull().sum(), "\n")
    print(3*"\n")

data_eda(train_data, "Train")
#data_eda(test_data, "Test")

##### Train Data EDA #####
Columns:
 Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='object') 

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   obj

In [19]:
print(train_data.isnull().sum() / len(train_data))

PassengerId     0.000000
HomePlanet      0.023122
CryoSleep       0.024963
Cabin           0.022892
Destination     0.020936
Age             0.020591
VIP             0.023352
RoomService     0.020821
FoodCourt       0.021051
ShoppingMall    0.023927
Spa             0.021051
VRDeck          0.021627
Name            0.023007
Transported     0.000000
dtype: float64


## Feature Engineering

In [20]:
# Create cabin section based features
train_data["CabinDeck"] = train_data["Cabin"].str.split("/").str[0]
train_data["CabinSide"] = train_data["Cabin"].str.split("/").str[2]

# Map boolean features into 0,1
train_data["CryoSleep"] = train_data["CryoSleep"].map({False: 0, True: 1, np.nan: 0})
train_data["VIP"] = train_data["VIP"].map({False: 0, True: 1, np.nan: 0})

# Map numerical NaN to 0
train_data = train_data.fillna(
    {
        "RoomService": 0,
        "FoodCourt": 0,
        "ShoppingMall": 0,
        "Spa": 0,
        "VRDeck": 0,
    }
)

# Combining billing 
train_data["Bills"] = train_data[["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]].sum(axis=1)

# Billing tier
threshold = 2160 # 1.5*IQR
train_data["BillingTier"] = pd.cut(
    train_data["Bills"],
    bins=[-np.inf,0,threshold,np.inf],
    labels=[0,1,2],
    right=True
).astype(int)

# Drop redundent features
drop_features = ["PassengerId", "Name", "Cabin", "RoomService","FoodCourt","ShoppingMall","Spa","VRDeck","Bills"]
train_data = train_data.drop(drop_features, axis=1)

In [21]:
train_data.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,Transported,CabinDeck,CabinSide,BillingTier
0,Europa,0,TRAPPIST-1e,39.0,0,False,B,P,0
1,Earth,0,TRAPPIST-1e,24.0,0,True,F,S,1
2,Europa,0,TRAPPIST-1e,58.0,1,False,A,S,2
3,Europa,0,TRAPPIST-1e,33.0,0,False,A,S,2
4,Earth,0,TRAPPIST-1e,16.0,0,True,F,S,1


## Data preprocessing function

In [22]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:

    # Create cabin section based features
    df["CabinDeck"] = df["Cabin"].str.split("/").str[0]
    df["CabinSide"] = df["Cabin"].str.split("/").str[2]

    # Map boolean features into 0,1
    df["CryoSleep"] = df["CryoSleep"].map({False: 0, True: 1, np.nan: 0})
    df["VIP"] = df["VIP"].map({False: 0, True: 1, np.nan: 0})

    # Map numerical NaN to 0
    df = df.fillna(
        {
            "RoomService": 0,
            "FoodCourt": 0,
            "ShoppingMall": 0,
            "Spa": 0,
            "VRDeck": 0,
        }
    )

    # Combining billing 
    df["Bills"] = df[["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]].sum(axis=1)

    # Billing tier
    threshold = 2160 # 1.5*IQR
    df["BillingTier"] = pd.cut(
        df["Bills"],
        bins=[-np.inf,0,threshold,np.inf],
        labels=[0,1,2],
        right=True
    ).astype(int)

    # Drop redundent features
    drop_features = ["PassengerId", "Name", "Cabin", "RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
    df = df.drop(drop_features, axis=1)

    return df

In [23]:
X_test = preprocess_data(test_data)
print(X_test.isnull().sum() / len(X_test))
X_test.head()

HomePlanet     0.020341
CryoSleep      0.000000
Destination    0.021510
Age            0.021277
VIP            0.000000
CabinDeck      0.023381
CabinSide      0.023381
Bills          0.000000
BillingTier    0.000000
dtype: float64


,HomePlanet,CryoSleep,Destination,Age,VIP,CabinDeck,CabinSide,Bills,BillingTier
0,Earth,1,TRAPPIST-1e,27.0,0,G,S,0.0,0
1,Earth,0,TRAPPIST-1e,19.0,0,F,S,2832.0,2
2,Europa,1,55 Cancri e,31.0,0,C,S,0.0,0
3,Europa,0,TRAPPIST-1e,38.0,0,C,S,7418.0,2
4,Earth,0,TRAPPIST-1e,20.0,0,F,S,645.0,1


In [24]:
X_train, y_train = train_data.drop(["Transported"], axis=1), train_data["Transported"]

## Preprocessing & Imputation

In [25]:

numerical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, ["Age"]),
    ("categorical", categorical_pipeline, ["HomePlanet","Destination","CabinDeck","CabinSide"]),
])

## Model fit

In [26]:
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])


# Fit model
model.fit(X_train, y_train)

# Train accuracy
train_accuracy = model.score(X_train, y_train)
print(f"Train accuracy: {train_accuracy:.4f}")

Train accuracy: 0.7446


## Model Prediction

In [27]:
# Process test data
preprocessed_test_data = preprocess_data(test_data)

prediction = model.predict(X_test)

In [28]:
# Save predictions
submission = pd.DataFrame({"PassengerId": test_data["PassengerId"], "Transported": prediction})
submission.to_csv("submission.csv", index=False)